# LLMBackend Usage

`LLMBackend` implements krrood's `GenerativeBackend` — swapping it onto
`context.query_backend` changes how a `Match` expression gets resolved.

Four patterns:
1. **`resolve_match()`** — Role 2 entry point; hides all krrood wiring
2. **Per-plan wiring** — set backend manually before each `execute_single`
3. **Context-level** — set once at construction; all plans use LLM
4. **Mid-session swap** — mix `ProbabilisticBackend` and `LLMBackend`


In [2]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
groundable_type = Body  # optional — defaults to Symbol inside llmr


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")
from llmr.reasoning.llm_config import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini")

---
## 1 — `resolve_match()` — Role 2 entry point

The cleanest way to use `LLMBackend` as a pure parameter resolver.
No `context.query_backend` wiring, no `execute_single` import — krrood plumbing
is hidden inside `resolve_match()`.  `instruction` is optional.


In [4]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core.pick_up import PickUpAction

from llmr import resolve_match,resolve_params

INSTRUCTION = "pick up the milk from the table"

match = underspecified(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=...,
)

# Role 2 — LLM fills all free slots from world context
plan_node = resolve_match(
    match           = match,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = INSTRUCTION,   # optional — helps with entity grounding
)
print("Backend type :", type(context.query_backend).__name__)
print("Plan node    :", plan_node)


Backend type : LLMBackend
Plan node    : PickUpAction


In [5]:
resolved_action = resolve_params(
    match           = match,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = INSTRUCTION,   # optional — helps with entity grounding
)

In [6]:
resolved_action

PickUpAction(object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('ab65705c-0dd7-4a11-9014-3ef7af9dfe93'), index=219), arm=RIGHT, grasp_description=GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('6121fea3-b362-4685-8813-6e5c55734965'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('4921a3e5-7b08-4456-98d7-7dbb36bf7d1d'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('b275cf64-a364-4195-bfdc-22274d51e8fe'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('e37e75b9-39ce-45dd-b130-c10539f6698f'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('6fe76ca5-8add-45b4-80e3-3a150380998e'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_stereo_optic

In [7]:
plan_node.__dict__

{'plan': Plan with 1 nodes,
 'status': <TaskStatus.CREATED: 0>,
 'start_time': datetime.datetime(2026, 4, 14, 13, 38, 43, 868711),
 'end_time': None,
 'reason': None,
 'result': None,
 'underspecified_action': Match(PickUpAction),
 '_action_iterator': None,
 'index': 0,
 'layer_index': 0}

In [ ]:
plan_node.perform()

In [ ]:
plan_node.plan.plan_graph.nodes()

---
## 2 — Context-level

Pass `LLMBackend` at context construction. Every plan in this context goes
through the LLM — no per-plan wiring needed.  Update `instruction` per-plan.


In [9]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.datastructures.dataclasses import Context

from llmr.backend import LLMBackend
from llmr.pycram_bridge import execute_single

llm_context = Context.from_world(
    world,
    query_backend=LLMBackend(
        llm=llm,
        groundable_type=groundable_type,
        instruction="go to the kitchen table",
    )
)

print("Backend type:", type(llm_context.query_backend).__name__)


Backend type: LLMBackend


In [ ]:
# Plan 1 — uses the backend set on llm_context
nav_match = underspecified(NavigateAction)(target_location=...)
nav_node = execute_single(nav_match, llm_context)
nav_node.perform()

In [ ]:
# Plan 2 — update instruction for the next action, same context
llm_context.query_backend.instruction = "pick up the milk from the table"

pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, llm_context)
pick_node.perform()

---
## 3 — Mid-session swap

Start with `ProbabilisticBackend` for routine actions. Switch to `LLMBackend`
via `resolve_match()` for novel underspecified actions, then restore.


In [ ]:
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core import PickUpAction, PlaceAction

from llmr.pycram_bridge import execute_single
from llmr.backend import LLMBackend

# Session starts with probabilistic backend
prob_backend = ProbabilisticBackend()
context.query_backend = prob_backend
print("Start  — backend:", type(context.query_backend).__name__)


In [ ]:
# Routine pick — handled by probabilistic backend
pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, context)
pick_node.perform()

In [ ]:
from krrood.entity_query_language.factories import underspecified
from pycram.robot_plans.actions.core import PlaceAction

from llmr import resolve_match

# Novel action — use resolve_match (Role 2 entry point)
place_match = underspecified(PlaceAction)(object_designator=..., target_location=...)

place_node = resolve_match(
    match           = place_match,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
    instruction     = "put the milk gently inside the fridge on the top shelf",
)
print("Swapped — backend:", type(context.query_backend).__name__)
place_node.perform()

# Restore probabilistic backend
context.query_backend = prob_backend
print("Restored — backend:", type(context.query_backend).__name__)


In [ ]:
# Restore probabilistic backend for remaining session
context.query_backend = prob_backend
print("Restored — backend:", type(context.query_backend).__name__)